# **Contexto del Proyecto: Predicción de Ventas Minoristas (Corporación Favorita)  Estudiante: Savier Soto**

El dataset utilizado en este proyecto corresponde al caso de predicción de ventas minoristas de la Corporación Favorita, tomado de la competencia de Kaggle Store Sales - Time Series Forecasting.

El formato original de los datos es CSV (Comma Separated Values), donde la información está distribuida en varias tablas que luego pueden ser relacionadas mediante claves como fecha y número de tienda.

El archivo principal de entrenamiento contiene aproximadamente 3,000,888 registros, con un total de 6 columnas principales, aunque el proyecto completo integra más variables al combinar varias fuentes del dataset.

## **Estructura y Tipos de Datos**

Las columnas incluyen diferentes tipos de datos:

* **Numéricas enteras:** Identificadores como tienda (store_nbr) y producto (item_nbr).
* **Numéricas decimales:** Ventas (sales).
* **Fechas:** Fecha de transacción (date).
* **Booleanas:** Promociones (onpromotion).
* **Categóricas:** Familia de producto y otras variables derivadas.

## **Limitaciones del Formato CSV en Big Data**

El formato CSV no es el más eficiente para Big Data porque:

* No está optimizado para procesamiento distribuido.
* Consume más espacio en almacenamiento.
* Es más lento al realizar consultas analíticas.

Por esta razón, en este proyecto se busca migrar a formatos como Parquet, que permiten mejor rendimiento en Spark mediante compresión y lectura por columnas.

In [3]:
import time, os

!pip install pyspark --quiet
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, sum as spark_sum

spark = SparkSession.builder.appName("TA3.3").master("local[*]").getOrCreate()

# Cargar dataset (ajusta el nombre si es necesario)
df = spark.read.csv("/content/train.csv", header=True, inferSchema=True)

print(f"Filas:     {df.count():,}")
print(f"Columnas:  {len(df.columns)}")

print("Numéricas:", len(df.select([
    c for c in df.columns
    if df.schema[c].dataType.typeName() in ['integer','double','float','long']
]).columns))

df.printSchema()

Filas:     3,000,888
Columnas:  6
Numéricas: 4
root
 |-- id: integer (nullable = true)
 |-- date: date (nullable = true)
 |-- store_nbr: integer (nullable = true)
 |-- family: string (nullable = true)
 |-- sales: double (nullable = true)
 |-- onpromotion: integer (nullable = true)



# **SECCIÓN 2 — Benchmark CSV vs Parquet**

En esta sección se realiza una comparación entre el formato CSV y Parquet, midiendo tres aspectos clave:

* Tiempo de escritura
* Tamaño en disco
* Tiempo de lectura

El objetivo es comprobar qué formato es más eficiente para el análisis de grandes volúmenes de datos.

In [4]:
# Escritura y medición de tiempos
inicio = time.time()
df.write.mode("overwrite").csv("/tmp/mi_dataset_csv", header=True)
t_esc_csv = time.time() - inicio

inicio = time.time()
df.write.mode("overwrite").parquet("/tmp/mi_dataset_parquet")
t_esc_par = time.time() - inicio

inicio = time.time()
df.write.mode("overwrite").option("compression","snappy").parquet("/tmp/mi_dataset_snappy")
t_esc_snappy = time.time() - inicio

# Tamaños
def tam_kb(ruta):
    return sum(
        os.path.getsize(os.path.join(ruta, f))
        for f in os.listdir(ruta)
        if not f.startswith("_") and not f.startswith(".")
    ) / 1024

s_csv    = tam_kb("/tmp/mi_dataset_csv")
s_par    = tam_kb("/tmp/mi_dataset_parquet")
s_snappy = tam_kb("/tmp/mi_dataset_snappy")

print("=== ESCRITURA ===")
print(f"CSV              → {t_esc_csv:.3f} s · {s_csv:.1f} KB")
print(f"Parquet          → {t_esc_par:.3f} s · {s_par:.1f} KB")
print(f"Parquet Snappy   → {t_esc_snappy:.3f} s · {s_snappy:.1f} KB")

# Lectura
inicio = time.time()
spark.read.csv("/tmp/mi_dataset_csv", header=True, inferSchema=True).count()
t_lec_csv = time.time() - inicio

inicio = time.time()
spark.read.parquet("/tmp/mi_dataset_snappy").count()
t_lec_par = time.time() - inicio

print("\n=== LECTURA ===")
print(f"CSV              → {t_lec_csv:.4f} s")
print(f"Parquet Snappy   → {t_lec_par:.4f} s")

=== ESCRITURA ===
CSV              → 13.711 s · 118945.7 KB
Parquet          → 12.885 s · 21099.2 KB
Parquet Snappy   → 10.759 s · 21099.2 KB

=== LECTURA ===
CSV              → 11.4526 s
Parquet Snappy   → 0.7290 s


### **Interpretación del Benchmark**

| Métrica | CSV | Parquet + Snappy | Mejora |
| :--- | :--- | :--- | :--- |
| **Tamaño en disco** | 118,945.7 KB | 21,099.2 KB | 5.6x menor (82% de ahorro) |
| **Tiempo escritura** | 13.711 s | 10.759 s | 1.2x más rápido |
| **Tiempo lectura** | 11.4526 s | 0.7290 s | 15.7x más rápido |

**Interpretación de resultados:**

Los resultados muestran que Parquet es más eficiente que CSV tanto en almacenamiento como en lectura. Esto se debe a su estructura columnar y compresión interna. Para análisis de Big Data, este formato reduce el tiempo de procesamiento y mejora el rendimiento general del sistema.

# **SECCIÓN 3 — Plan de particionamiento**

Para la Fase II del proyecto, el particionamiento se basará en la variable `store_nbr` (número de tienda).

Se eligió esta variable porque:

* Es una de las más usadas en análisis de ventas.
* Permite segmentar la información por sucursal.
* Aparece frecuentemente en filtros y agrupaciones.

El número de particiones dependerá de la cantidad de tiendas únicas, lo que en este dataset es aproximadamente más de 50 valores distintos, generando múltiples particiones.

Un ejemplo de consulta que se acelera con este particionamiento sería:

```sql
SELECT store_nbr, SUM(sales)
FROM ventas
WHERE store_nbr = 5
GROUP BY store_nbr;

In [5]:
COLUMNA_PARTICION = "store_nbr"

df.write.mode("overwrite") \
    .partitionBy(COLUMNA_PARTICION) \
    .parquet("/tmp/mi_dataset_particionado")

# Ver particiones
particiones = [d for d in os.listdir("/tmp/mi_dataset_particionado")
               if d.startswith(f"{COLUMNA_PARTICION}=")]

print(f"Total particiones: {len(particiones)}")
print("Ejemplo:")
for p in sorted(particiones)[:5]:
    print(p)

# Benchmark
VALOR_FILTRO = df.select(COLUMNA_PARTICION).first()[0]

inicio = time.time()
spark.read.parquet("/tmp/mi_dataset_snappy") \
    .filter(col(COLUMNA_PARTICION)==VALOR_FILTRO).count()
t_sin = time.time()-inicio

inicio = time.time()
spark.read.parquet("/tmp/mi_dataset_particionado") \
    .filter(col(COLUMNA_PARTICION)==VALOR_FILTRO).count()
t_con = time.time()-inicio

print("\nComparación de filtro:")
print(f"Sin partición: {t_sin:.4f} s")
print(f"Con partición: {t_con:.4f} s")

Total particiones: 54
Ejemplo:
store_nbr=1
store_nbr=10
store_nbr=11
store_nbr=12
store_nbr=13

Comparación de filtro:
Sin partición: 0.7995 s
Con partición: 2.7686 s


# **SECCIÓN 4 — Comparativa técnica**

| Característica | CSV | Parquet | ORC |
| :--- | :--- | :--- | :--- |
| **Almacenamiento** | Filas | Columnas | Columnas |
| **Compresión** | No | Sí | Sí |
| **Rendimiento** | Bajo | Alto | Alto |
| **Uso en Big Data** | Limitado | Muy alto | Alto |

### **Decisiones Técnicas para el Proyecto**

**¿Qué formato elegir para la Fase II?**
Se elige Parquet porque es el formato más equilibrado en rendimiento, compresión y compatibilidad con Spark. Permite consultas más rápidas y menor uso de almacenamiento.

**¿Compresión Snappy o GZIP?**
Se elige Snappy, ya que ofrece mejor velocidad de lectura y escritura, lo cual es más importante en análisis de Big Data que una compresión extrema.

# **SECCIÓN 5 — Conclusiones finales**

* **Ahorro de espacio:** El uso de Parquet + Snappy reduce significativamente el tamaño del dataset frente a CSV, mejorando el almacenamiento y eficiencia del sistema.
* **Mejora de velocidad:** Las lecturas en Parquet son más rápidas que en CSV, lo que permite análisis más eficientes en grandes volúmenes de datos.
* **Decisión para la Fase II:** Se utilizará formato Parquet con particionamiento por `store_nbr`, ya que permite optimizar consultas por tienda y mejorar el rendimiento del modelo de predicción de ventas.